In [51]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [52]:
! pip install -q chonkie sentence-transformers faiss-cpu jsonlines

In [157]:
from kaggle_secrets import UserSecretsClient
import wandb

from chonkie import TokenChunker

import numpy as np              
import pandas as pd             
import matplotlib.pyplot as plt 
import seaborn as sns           
import torch                    
import torch.nn as nn         
from torch.utils.data import Dataset,DataLoader
from collections import Counter 
from string import punctuation  
import warnings                 
import string
import re
import unicodedata

from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder 
import faiss 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm
import uuid
import jsonlines

from datasets import load_dataset

In [54]:
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch Version: 2.10.0+cu128
NumPy Version: 2.4.6
Pandas Version: 2.3.3
CUDA Available: True
CUDA Version: 12.8
GPU Device: Tesla T4


# W&B

In [55]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api")

In [56]:
wandb.login(key=wandb_key)
wapi=wandb.Api()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [57]:
CONFIG={
    "project_name":"23f2000391-t22026",
    "embedding_model":"all-MiniLM-L6-v2",
    "knowledge_base":"train.csv(Preprocessed Prompts and Choices)",
    "search_index":"IndexFlatIP",
    "cross_encoder":"ms-marco-MiniLM-L-6-v2",
    "retrieval_k":50,
    "split":"Stratified-K-Fold",
    "folds_num":5
}

In [ ]:
wandb.init(
    project=CONFIG["project_name"],
    name='RAG System(With Preprocessed Train Data)',
    config=CONFIG
)


# Dataset

In [58]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [59]:
train.drop(columns='id',inplace=True)
train.head()

,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [60]:
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [61]:
test.drop(columns='id',inplace=True)
test.head()

,prompt,A,B,C,D,E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [62]:
GLOBAL_choices=["A","B","C","D","E"]

# EDA

In [63]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   prompt  2000 non-null   object
 1   A       2000 non-null   object
 2   B       2000 non-null   object
 3   C       2000 non-null   object
 4   D       2000 non-null   object
 5   E       2000 non-null   object
 6   answer  2000 non-null   object
dtypes: object(7)
memory usage: 109.5+ KB


In [64]:
train.describe()

,prompt,A,B,C,D,E,answer
count,2000,2000,2000,2000,2000,2000,2000
unique,1758,316,328,303,318,320,5
top,Choose the correct answer: What is the main se...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,B
freq,4,21,21,21,21,21,490


There might be some duplicates since only 1758 out of the 2000 prompts are unique

## Duplicates

In [65]:
print(f"Number of Duplicates: {train.duplicated().sum()}")

Number of Duplicates: 183


In [ ]:
'''print("Shape before dropping duplicates",train.shape)
train.drop_duplicates(inplace=True)
print("Shape after dropping duplicates",train.shape)'''

In [66]:
train.duplicated(subset=['prompt','answer']).sum()

np.int64(242)

In [67]:
print("Number of Duplicated Rows where only one option is changed\n")
for opt in ['A','B','C','D','E']:
    cols=[i for i in train.columns if i!=opt]
    print(f"Option {opt}: {train.duplicated(subset=cols).sum()}")

Number of Duplicated Rows where only one option is changed

Option A: 185
Option B: 189
Option C: 185
Option D: 188
Option E: 184


In [68]:
train[train['prompt'].duplicated()].sort_values(['prompt'])[:5]

,prompt,A,B,C,D,E,answer
605,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
456,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
1464,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
1621,Choose the correct answer: What are the four q...,"Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...","Holocrystalline, hypohyaline, hypercrystalline...","Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...",B
944,Choose the correct answer: What are the four q...,"Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...","Holocrystalline, hypohyaline, hypercrystalline...","Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...",B


In [69]:
for i in train.columns:
    print(i+": "+train.loc[1944][i])

prompt: Choose the correct answer: What is a planetary mechanism? based on the given context.
A: A mechanism of planets that are all located in the same solar mechanism.
B: A framework of planets that are all the same size and shape.
C: Any set of gravitationally bound non-stellar objects in or out of orbit around a star or star structure.
D: A framework of planets that are all located in the same galaxy.
E: A framework of planets that are all made of gas.
answer: C


In [70]:
for i in train.columns:
    print(i+": "+train.loc[545][i])

prompt: Choose the correct answer: What is a planetary structure? based on the given context.
A: A mechanism of planets that are all located in the same solar mechanism.
B: A mechanism of planets that are all the same size and shape.
C: Any set of gravitationally bound non-stellar objects in or out of orbit around a star or star structure.
D: A framework of planets that are all located in the same galaxy.
E: A framework of planets that are all made of gas.
answer: C


In [71]:
train.groupby(["prompt","answer"]).agg(prompts=("prompt", list),count=("prompt", "size"))

,,prompts,count
prompt,answer,,
"Choose the correct answer: How do the Lunar Laser Ranging Experiment, radar astronomy, and the Deep Space Network determine distances to the Moon, planets, and spacecraft? among the listed options.",B,[Choose the correct answer: How do the Lunar L...,1
"Choose the correct answer: How do the Lunar Laser Ranging Experiment, radar astronomy, and the Deep Space Network determine distances to the Moon, planets, and spacecraft? based on the given context.",B,[Choose the correct answer: How do the Lunar L...,1
Choose the correct answer: How many crystallographic point groups are there in three-dimensional space? among the listed options.,B,[Choose the correct answer: How many crystallo...,1
Choose the correct answer: How many crystallographic point groups are there in three-dimensional space? based on the given context.,B,[Choose the correct answer: How many crystallo...,1
Choose the correct answer: What are permutation-inversion groups?,E,[Choose the correct answer: What are permutati...,3
...,...,...,...
Who published the first theory that was able to encompass previously separate field theories to provide a unifying theory of electromagnetism?,A,[Who published the first theory that was able ...,1
Who shared the other half of the Nobel Prize with Yoichiro Nambu for discovering the origin of the explicit breaking of CP symmetry in the weak interactions?,B,[Who shared the other half of the Nobel Prize ...,1
Who was Giordano Bruno?,D,[Who was Giordano Bruno?],1


In [72]:
train[train.duplicated(subset=GLOBAL_choices)].sort_values(GLOBAL_choices)

,prompt,A,B,C,D,E,answer
1112,Pick the best possible answer: What is the dec...,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E
1145,What is the decay energy for the free neutron ...,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E
1171,Choose the correct answer: What is the decay e...,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E
1208,Pick the best possible answer: What is the dec...,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E
1683,Pick the best possible answer: What is the dec...,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E
...,...,...,...,...,...,...,...
1175,Determine the correct option: What is the piez...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1224,Determine the correct option: What is the piez...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1293,Which of the following is correct? What is the...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1346,Identify the correct statement: What is the pi...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B


In [73]:
train.duplicated(subset=GLOBAL_choices).sum()

np.int64(1412)

In [74]:
grouped_train=train.groupby(["A", "B", "C", "D", "E","answer"]).agg(prompts=("prompt", list),count=("prompt", "size")).reset_index()
grouped_train.head()

,A,B,C,D,E,answer,prompts,count
0,0.013343 MeV,0.013 MeV,"1,000 MeV",0.782 MeV,0.782343 MeV,E,[Pick the best possible answer: What is the de...,8
1,7,32,14,5,27,B,[Determine the correct option: How many crysta...,16
2,8 times more than the Sun,8 times less than the Sun,13 light years away from Earth,Unknown,Equal to the Sun,B,[Pick the best possible answer: What is the me...,12
3,A German philosopher who supported the Kepleri...,An English philosopher who supported the Ptole...,A French philosopher who supported the Aristot...,An Italian philosopher who supported the Coper...,A Spanish philosopher who supported the Galile...,D,[Which of the following is correct? Who was Gi...,1
4,A German philosopher who supported the Kepleri...,An English philosopher who supported the Ptole...,A French philosopher who supported the Aristot...,An Italian philosopher who supported the Coper...,A Spanish philosopher who supported the Galile...,D,[Select the most accurate option: Who was Gior...,1


In [75]:
print("Number of Unique Questions",grouped_train.shape[0])

Number of Unique Questions 588


In [76]:
inst=set()
for _,row in train.iterrows():
    text=row['prompt']
    prefix=text.index(':') if ':' in text else 0
    suffix=text.rfind('?')+1 if '?' in text else len(text)
    if prefix!=0:
        inst.add(text[:prefix])
    if suffix!=len(text):
        inst.add(text[suffix:])
inst

{' among the listed options.',
 ' based on the given context.',
 ' carefully.',
 ' from the following choices.',
 'Choose the correct answer',
 'Determine the correct option',
 'Identify the correct statement',
 'Pick the best possible answer',
 'Select the most accurate option',
 "What is the purpose of expressing a map's scale as a ratio, such as 1",
 "Which of the following is correct? What is the purpose of expressing a map's scale as a ratio, such as 1"}

In [77]:
test.duplicated().sum()

np.int64(7)

In [78]:
inst_test=set()
for _,row in test.iterrows():
    text=row['prompt']
    prefix=text.index(':') if ':' in text else 0
    suffix=text.rfind('?')+1 if '?' in text else len(text)
    if prefix!=0:
        inst_test.add(text[:prefix])
    if suffix!=len(text):
        inst_test.add(text[suffix:])
inst_test

{' among the listed options.',
 ' based on the given context.',
 ' carefully.',
 ' from the following choices.',
 'Choose the correct answer',
 'Determine the correct option',
 'Identify the correct statement',
 'Pick the best possible answer',
 'Select the most accurate option',
 "What is the purpose of expressing a map's scale as a ratio, such as 1",
 "Which of the following is correct? What is the purpose of expressing a map's scale as a ratio, such as 1"}

## Class Distribution

In [79]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

## Prompt Word Length

In [80]:
prompt_word_length=train['prompt'].apply(lambda x:len(x.split()))
prompt_word_length.describe()

count    2000.00000
mean       18.14650
std         6.78189
min         3.00000
25%        14.00000
50%        17.00000
75%        22.00000
max        51.00000
Name: prompt, dtype: float64

In [81]:
prompt_char_len=train["prompt"].str.len()
prompt_char_len.describe()

count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64

## Options Word Length

In [82]:
word_lens=[]
for col in GLOBAL_choices:
    word_lens.append(train[col].str.split().str.len())

In [83]:
for i in range(len(word_lens)):
    print(f"Option {chr(i+ord("A"))}")
    print(word_lens[i].describe())

Option A
count    2000.000000
mean       26.146000
std        17.249651
min         1.000000
25%        13.000000
50%        22.000000
75%        37.000000
max        81.000000
Name: A, dtype: float64
Option B
count    2000.000000
mean       26.516000
std        18.653893
min         1.000000
25%        12.000000
50%        23.000000
75%        36.000000
max       118.000000
Name: B, dtype: float64
Option C
count    2000.00000
mean       26.54950
std        17.20987
min         1.00000
25%        14.75000
50%        24.00000
75%        36.00000
max        82.00000
Name: C, dtype: float64
Option D
count    2000.000000
mean       25.999500
std        17.098909
min         1.000000
25%        15.000000
50%        22.000000
75%        36.000000
max        78.000000
Name: D, dtype: float64
Option E
count    2000.000000
mean       26.187000
std        17.852658
min         1.000000
25%        13.000000
50%        23.000000
75%        36.000000
max       105.000000
Name: E, dtype: float64


## Correct Answer Word Length

In [84]:
correct_lengths=[]
for _, row in train.iterrows():
    correct_lengths.append(len(row[row["answer"]].split()))

pd.Series(correct_lengths).describe()

count    2000.000000
mean       28.659000
std        18.331005
min         1.000000
25%        16.000000
50%        28.000000
75%        40.000000
max       105.000000
dtype: float64

## Categories of Questions

In [ ]:
'''zs=pipeline("zero-shot-classification")
zs'''

In [ ]:
'''single_question=grouped_train.reset_index()['prompts'].apply(lambda x:x[0])
single_question'''

In [ ]:
'''results = zs(
    single_question.tolist(),
    candidate_labels=["Physics","Chemistry","Biology","Mathematics","Computer Science","Engineering","Medicine","Astronomy","Philosophy","History","Economics","Politics","Geography","Language and Literature","Religion","General Knowledge"],
    batch_size=16,
    multi_label=False
)

cats=[result["labels"][0] for result in results]
cats[:5]'''

In [ ]:
'''grouped_train["category"]=cats
grouped_train["category"].value_counts()'''

# Preprocessing

In [85]:
def clean_text(text):
    text=str(text)
    text=unicodedata.normalize("NFKC", text)
    text=text.strip()
    text=re.sub(r"\s+", " ", text)
    return text

In [86]:
PREFIXES=[
    "Choose the correct answer:",
    "Pick the best possible answer:",
    "Determine the correct option:",
    "Select the most accurate option:",
    "Identify the correct statement:",
    "Which of the following is correct?"
]

SUFFIXES=[
    "based on the given context.",
    "from the following choices.",
    "among the listed options.",
    "carefully."
]

def clean_prompt(text):
    text=text.strip()

    for p in PREFIXES:
        if text.startswith(p):
            text=text[len(p):].strip()

    for s in SUFFIXES:
        if text.endswith(s):
            text=text[:-len(s)].strip()

    return text

In [87]:
for col in ["prompt", "A", "B", "C", "D", "E"]:
    train[f"clean_{col}"] = train[col].apply(clean_text)
    test[f"clean_{col}"] = test[col].apply(clean_text)

train["clean_prompt"]=train["clean_prompt"].apply(clean_prompt)
test["clean_prompt"]=test["clean_prompt"].apply(clean_prompt)

In [88]:
train.head()

,prompt,A,B,C,D,E,answer,clean_prompt,clean_A,clean_B,clean_C,clean_D,clean_E
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,What is Martin Heidegger's view on the relatio...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,What is the term used in astrophysics to descr...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,What is Martin Heidegger's view on the relatio...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,What is the concept of simultaneity in Einstei...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...


In [89]:
test.head()

,prompt,A,B,C,D,E,clean_prompt,clean_A,clean_B,clean_C,clean_D,clean_E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",What is the relationship between the Hamiltoni...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...","What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,What is the reason for the sun appearing sligh...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [90]:
train["clean_combined_text"]=(
    "Question: "+train["clean_prompt"] +
    "\nA: "+train["clean_A"] +
    "\nB: "+train["clean_B"] +
    "\nC: "+train["clean_C"] +
    "\nD: "+train["clean_D"] +
    "\nE: "+train["clean_E"]
)

print(train["clean_combined_text"][0])

Question: What is Martin Heidegger's view on the relationship between time and human existence?
A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.
D: Martin Heidegger believes that the relationship between time and human existence is cyclical. The past

In [93]:
count=0
preds=[]
l=[]
l1=[]
for _,row in test.iterrows():
    if row['clean_prompt'] in train['clean_prompt'].tolist():
        if (row.drop('prompt')==train.iloc[train['clean_prompt'].tolist().index(row['clean_prompt'])].drop(['prompt','answer','clean_combined_text'])).all():
            preds.append(train.iloc[train['clean_prompt'].tolist().index(row['clean_prompt'])]['answer'])
            count+=1
        else:
            l.append(row)
            l1.append(train.iloc[train['clean_prompt'].tolist().index(row['clean_prompt'])])
print(count)

392


In [96]:
for i in GLOBAL_choices:
    print(i+": "+l[1][i])

A: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with different gases at equal temperatures. The demon selectively allows molecules to pass from one side to the other, causing an increase in temperature in one part and a decrease in temperature in the other, contrary to the second law of thermodynamics.
B: A thought experiment in which a demon guards a macroscopic trapdoor in a wall separating two parts of a container filled with different gases at different temperatures. The demon selectively allows molecules to pass from one side to the other, causing a decrease in temperature in one part and an increase in temperature in the other, in accordance with the second law of thermodynamics.
C: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pa

In [99]:
for i in GLOBAL_choices:
    print(i+": "+l1[1][i])
print("answer: "+l1[1]["answer"])

A: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with different gases at equal temperatures. The demon selectively allows molecules to pass from one side to the other, causing an raise in temperature in one part and a lower in temperature in the other, contrary to the second law of thermodynamics.
B: A thought experiment in which a demon guards a macroscopic trapdoor in a wall separating two parts of a container filled with different gases at different temperatures. The demon selectively allows molecules to pass from one side to the other, causing a minimize in temperature in one part and an boost in temperature in the other, in accordance with the second law of thermodynamics.
C: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from o

## Corpus for vocabulary

In [100]:
text=' '.join(train['clean_combined_text'].tolist())
words=text.split()
print("Total words in corpus:",len(words))
print("Total unique words in corpus:",len(set(words)))

Total words in corpus: 297463
Total unique words in corpus: 3944


In [101]:
word_freq=Counter(words)
print(f"Most Frequent Words:")
for i,j in (word_freq.most_common(10)):
    print(f"{i}: {j}")
print(f"\nLeast Frequent Words:")
for i,j in (word_freq.most_common()[-11:-1][::-1]):
    print(f"{i}: {j}")

Most Frequent Words:
the: 19901
of: 14324
a: 10795
is: 9900
and: 6917
in: 6581
to: 6003
that: 4348
The: 4332
by: 2153

Least Frequent Words:
decrease,: 1
subframeworks: 1
subsystem.: 1
subsystems: 1
subsystem: 1
theory,: 1
essential.: 1
reduce,: 1
increases: 1
important.: 1


## Split

In [102]:
#train_df,val_df=train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=42)

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#print(train_df["answer"].value_counts()/len(train_df))
#print(val_df["answer"].value_counts()/len(val_df))

# Scoring

In [103]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds,start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [104]:
def top3_accuracy(y_true, predictions):
    return np.mean([truth in pred for truth, pred in zip(y_true, predictions)])

In [105]:
def top1_accuracy(y_true, predictions):
    top1=[pred[0] if len(pred) else "Z" for pred in predictions]

    return accuracy_score(y_true,top1)

In [158]:
def macro_f1_score(y_true, predictions):
    top1_preds=[]

    for pred in predictions:
        top1_preds.append(pred[0])

    return f1_score(y_true,top1_preds,average="macro",labels=["A", "B", "C", "D", "E"],zero_division=0)

# PreTrained Models

## Zero-Shot Classification

In [ ]:
'''qwen_model_name="Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer=AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model=AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16,
    device_map="auto"
)'''

In [ ]:
'''def create_zero_shot_prompt(row):
    zero_shot_prompt=f"""
    You are solving a multiple choice question containing 5 choices.
    Question:
    {row["prompt"]}
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    Return ONLY the three most likely answer labels with a single space separating them.
    Example:
    C A D
    """
    return zero_shot_prompt'''

In [ ]:
'''def create_few_shot_prompt(row):
    few_shot_examples = []
    for label in GLOBAL_choices:
        example = train_df[train_df["answer"] == label].sample(n=1,random_state=42).iloc[0]
        few_shot_examples.append(example)
    prompt = """You are an expert at solving multiple-choice questions.
                Below are some solved examples.\n"""

    for i, ex in enumerate(few_shot_examples, 1):

        prompt += f"""Example {i}
        Question: {ex["prompt"]}
        
        Choices:
        A. {ex["A"]}
        B. {ex["B"]}
        C. {ex["C"]}
        D. {ex["D"]}
        E. {ex["E"]}
        
        Correct Answer:
        {ex["answer"]}
        
        """
        
    prompt += f"""
    Now answer the following question.
    
    Question:
    {row["prompt"]}
    
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    
    Rank the choices based on their probability of being the correct answer.
    Then return the top three answer labels separated by spaces.
    
    Example output:
    C A D
    
    Do not explain your answer.
    """

    return prompt'''

In [ ]:
'''def predict_qwen(row):

    prompt = create_few_shot_prompt(row)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(qwen_model.device)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    prediction = qwen_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    labels = re.findall(r"\b[A-E]\b", prediction)

    return labels[:3]'''

In [ ]:
'''predictions = []

fold_map = []
fold_acc = []
fold_top3 = []

for fold,(train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        pred = predict_qwen(row)
        predictions.append(pred)
        torch.cuda.empty_cache()
    map3 = map_at_3(
        val_df["answer"],
        predictions
    )
    
    acc = top1_accuracy(
        val_df["answer"],
        predictions
    )
    
    top3 = top3_accuracy(
        val_df["answer"],
        predictions
    )
    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Accuracy": acc,
        "Top3 Accuracy": top3
    })
    fold_map.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)
print(f"MAP@3 : {np.mean(fold_map):.4f}")
print(f"Accuracy : {np.mean(fold_acc):.4f}")
print(f"Top3 Accuracy : {np.mean(fold_top3):.4f}")'''

In [ ]:
'''wandb.log({
    "Average MAP@3": np.mean(fold_map),
    "Average Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

## RoBERTa

# Model from Scratch

## BiLSTM + Attention Scores

In [ ]:
'''tokenizer=AutoTokenizer.from_pretrained(CONFIG['tokenizer'])
tokenizer'''

In [ ]:
'''class Attention(nn.Module):
    def __init__(self,hidden_dim):
        super().__init__()
        self.linear=nn.Linear(hidden_dim*2,1)

    def forward(self,x):
        weights=torch.softmax(self.linear(x),dim=1)
        context=(weights*x).sum(dim=1)
        return context'''

In [ ]:
'''class BiLSTMAttention(nn.Module):
    def __init__(self,vocab_size,embedding_dim=300,hidden_dim=256,num_layers=2,dropout=0.3):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.lstm=nn.LSTM(embedding_dim,hidden_dim,num_layers=num_layers,batch_first=True,bidirectional=True,dropout=dropout)
        self.attention=Attention(hidden_dim)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_dim*2,1)

    def forward(self,input_ids):
        batch_size=input_ids.size(0)
        scores=[]
        for i in range(5):
            x=input_ids[:,i]
            x=self.embedding(x)
            output,_=self.lstm(x)
            context=self.attention(output)
            context=self.dropout(context)
            score=self.fc(context)
            scores.append(score.squeeze(1))
        scores=torch.stack(scores,dim=1)
        return scores'''

In [ ]:
'''def encode_question(question,option):
    encoding=tokenizer(
        question,
        option,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    return encoding["input_ids"]'''

In [ ]:
'''label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}'''

In [ ]:
'''class MCQDataset(Dataset):
    def __init__(self, dataframe):
        self.df=dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        inputs=[]
        for choice in GLOBAL_choices:
            ids=encode_question(row["prompt"],row[f"{choice}"])
            inputs.append(ids)

        inputs=torch.tensor(inputs)
        label=label2id[row["answer"]]
        return {"input_ids": inputs,"label": torch.tensor(label)}'''

In [ ]:
'''def train_one_epoch(model,dataloader,optimizer,criterion,device):
    model.train()
    running_loss=0

    for batch in tqdm(dataloader):
        input_ids=batch["input_ids"].to(device)
        labels=batch["label"].to(device)
        optimizer.zero_grad()
        outputs=model(input_ids)
        loss=criterion(outputs,labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

        optimizer.step()
        running_loss+=loss.item()

    epoch_loss=running_loss/len(dataloader)

    return epoch_loss'''

In [ ]:
'''def validate(model,dataloader,criterion,device):
    model.eval()
    running_loss=0
    predictions=[]
    ground_truth=[]
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids=batch["input_ids"].to(device)
            labels=batch["label"].to(device)

            outputs=model(input_ids)
            loss=criterion(outputs, labels)
            running_loss+=loss.item()

            probs=torch.softmax(outputs, dim=1)

            top3=torch.topk(probs,k=3,dim=1).indices.cpu().numpy()

            for pred in top3:
                predictions.append([id2label[i] for i in pred])

            ground_truth.extend([id2label[i.item()] for i in labels])

    val_loss=running_loss/len(dataloader)
    map3=map_at_3(ground_truth,predictions)
    acc=top1_accuracy(ground_truth,predictions)
    top3_acc=top3_accuracy(ground_truth,predictions)

    return (val_loss,map3,acc,top3_acc,predictions)'''

In [ ]:
'''fold_map3=[]
fold_top1=[]
fold_top3=[]
best_cv_map3=-1
for fold,(train_idx,val_idx) in enumerate(skf.split(train,train["answer"])):
    
    print(f"{'='*10}Fold {fold+1}{'='*10}")

    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)

    train_dataset=MCQDataset(train_df)
    val_dataset=MCQDataset(val_df)
    
    train_loader=DataLoader(train_dataset,batch_size=CONFIG["batch_size"],shuffle=True)
    val_loader=DataLoader(val_dataset,batch_size=CONFIG["batch_size"],shuffle=False)

    bilstm_model=BiLSTMAttention(
        vocab_size=tokenizer.vocab_size,
        embedding_dim=CONFIG["embedding_dim"],
        hidden_dim=CONFIG["hidden_dim"],
        num_layers=CONFIG["num_layers"],
        dropout=CONFIG["dropout"]
    )
    bilstm_model.to(device)
    
    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.AdamW(bilstm_model.parameters(),lr=CONFIG["lr"],weight_decay=CONFIG["weight_decay"])

    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=0.5,patience=2)
    
    best_map3=-1
    best_top3=0
    best_top1=0
    patience=3
    counter=0

    for epoch in range(CONFIG["epochs"]):

        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
        train_loss=train_one_epoch(bilstm_model,train_loader,optimizer,criterion,device)
        (val_loss,map3,acc,top3_acc,predictions)=validate(bilstm_model,val_loader,criterion,device)
    
        print(f"Train Loss:{train_loss:.4f}")
        print(f"Val Loss  :{val_loss:.4f}")
        print(f"MAP@3     :{map3:.4f}")
        print(f"Top1 Acc  :{acc:.4f}")
        print(f"Top3 Acc  :{top3_acc:.4f}")

        scheduler.step(map3)

        wandb.log({
            "Fold":fold+1,
            "Epoch":epoch+1,
            "Train Loss":train_loss,
            "Validation Loss":val_loss,
            "Validation MAP@3":map3,
            "Validation Top1 Accuracy":acc,
            "Validation Top3 Accuracy":top3_acc,
            "Learning Rate":optimizer.param_groups[0]["lr"]
            })

        if map3>best_map3:
            best_map3=map3
            best_top1=acc
            best_top3=top3_acc
            counter=0

            if best_map3>best_cv_map3:
                best_cv_map3=map3
                torch.save(bilstm_model.state_dict(),"bilstm_best_model.pt")
        else:
            counter+=1
            if counter>=patience:
                print("Early Stopping")
                break
    fold_map3.append(best_map3)
    fold_top1.append(best_top1)
    fold_top3.append(best_top3)
    
    wandb.log({
        "Fold":fold+1,
        "Fold MAP@3":best_map3,
        "Fold Top1":best_top1,
        "Fold Top3":best_top3
        })
print(f"MAP@3: {np.mean(fold_map3):.4f}")
print(f"Top1 : {np.mean(fold_top1):.4f}")
print(f"Top3 : {np.mean(fold_top3):.4f}")'''

In [ ]:
'''print(f"MAP@3: {np.mean(fold_map3):.4f}")
print(f"Top1 : {np.mean(fold_top1):.4f}")
print(f"Top3 : {np.mean(fold_top3):.4f}")

wandb.log({
    "MAP@3 Mean": np.mean(fold_map3),
    "Top1 Mean": np.mean(fold_top1),
    "Top3 Mean": np.mean(fold_top3),
})'''

In [ ]:
'''artifact=wandb.Artifact("bilstm_best_model",type="model")
artifact.add_file("/kaggle/working/bilstm_best_model.pt")
wandb.log_artifact(artifact)
wandb.finish()'''

In [ ]:
'''class TestDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        inputs = []
        for choice in GLOBAL_choices:
            ids = encode_question(row["prompt"],row[f"{choice}"])
            inputs.append(ids)

        return {"input_ids": torch.tensor(inputs)}'''

In [ ]:
'''test_dataset=TestDataset(test)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)'''

In [ ]:
'''bilstm_model = BiLSTMAttention(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=300,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

bilstm_model.load_state_dict(
    torch.load(
        "bilstm_best_model.pt",
        map_location=device
    )
)

bilstm_model.eval()'''

In [ ]:
'''def bilstm_predict(model,dataloader,device):
    model.eval()
    predictions=[]
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids=batch["input_ids"].to(device)
            outputs=model(input_ids)
            probs=torch.softmax(outputs,dim=1)

            top3=torch.topk(probs,k=3,dim=1).indices.cpu().numpy()

            for pred in top3:
                letters=[id2label[i] for i in pred]
                predictions.append(" ".join(letters))

    return predictions'''

In [ ]:
'''test_predictions=bilstm_predict(bilstm_model,test_loader,device)
test_predictions[:5]'''

# Model of Choice

## Tf-Idf + Logistic Regression

In [ ]:
'''def build_pairwise_dataset(df,test=False):
    rows=[]
    options=GLOBAL_choices
    for qid,row in df.iterrows():
        for c in options:
            if not test:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "label":1 if c==row["answer"] else 0,
                    "question_id":qid,
                    "option":c
                })
            else:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "question_id":qid,
                    "option":c
                })

    return pd.DataFrame(rows)'''

In [ ]:
'''vectorizer=TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=3,
    sublinear_tf=True
)
clf=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42)'''

In [ ]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    pair_train=build_pairwise_dataset(train_df)
    pair_val=build_pairwise_dataset(val_df)

    X_train=vectorizer.fit_transform(pair_train["text"])
    X_val=vectorizer.transform(pair_val["text"])

    clf.fit(X_train,pair_train["label"])
    
    probs=clf.predict_proba(X_val)[:,1]

    predictions=[]
    choices=GLOBAL_choices
    for i in range(len(val_df)):
        start=i*5
        end=start+5
        scores=probs[start:end]
        ranked=np.argsort(scores)[::-1]
        top3=[choices[j] for j in ranked[:3]]
        predictions.append(top3)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })'''

In [ ]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

In [ ]:
'''pair_train=build_pairwise_dataset(train)
X_train=vectorizer.fit_transform(pair_train["text"])
clf.fit(X_train,pair_train["label"])'''

In [ ]:
'''pair_test=build_pairwise_dataset(test,True)
X_test=vectorizer.transform(pair_test["text"])
probs=clf.predict_proba(X_test)[:, 1]
probs'''

In [ ]:
'''choices=GLOBAL_choices
test_predictions=[]
for i in range(len(test)):
    scores=probs[i*5:(i+1)*5]
    ranked=np.argsort(scores)[::-1]
    top3=[choices[j] for j in ranked[:3]]
    test_predictions.append(" ".join(top3))
test_predictions[:5]'''

## RAG System

### Wikepedia

In [ ]:
'''wiki=load_dataset("wikimedia/wikipedia","20231101.en",split="train")
wiki=wiki.select(range(50000))'''

In [ ]:
'''def clean_data(text):
    text=text.lower()
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"\[[^\]]*\]","",text)
    text=text.strip()
    return text'''

In [ ]:
'''chunker=TokenChunker(chunk_size=CONFIG["chunk_size"],chunk_overlap=CONFIG["chunk_overlap"])
chunker'''

In [ ]:
'''model = SentenceTransformer("BAAI/bge-base-en-v1.5") 
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",dtype=torch.float32, device_map="auto")'''

In [ ]:
'''kb=[]
chunk_records=[]
for article in wiki:
    title=article["title"]
    text=clean_data(article["text"])
    url=article.get("url", "")

    if len(text.strip()) == 0:
        continue

    chunks=chunker(text)

    for chunk_idx, chunk in enumerate(chunks):
        chunk_text=chunk.text
        kb.append(chunk_text)
        chunk_records.append({
            "chunk_id":str(uuid.uuid4()),
            "chunk_index": chunk_idx,
            "title": title,
            "url": url,
            "text": chunk_text,
            "token_count": chunk.token_count
        })

print(f"Total chunks generated: {len(chunk_records)}")'''

In [ ]:
'''with jsonlines.open('/kaggle/working/knowledge_base.jsonl', mode='w') as writer:
    writer.write_all(chunk_records)

wandb.config.update({"total_chunks": len(chunk_records)})

print(f"Chunk and metadata saved")'''

In [ ]:
'''texts_to_embed=[chunk["text"] for chunk in chunk_records]
embeddings = model.encode(
    texts_to_embed,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)'''

In [ ]:
'''index_path="/kaggle/working/vector_index.idx"

embedding_dimension=embeddings.shape[1]
faiss_index=faiss.IndexFlatL2(embedding_dimension) 
faiss_index.add(embeddings)

faiss.write_index(faiss_index, index_path)

wandb.config.update({"embedding_dimension": embedding_dimension})
print(f"Saved FAISS index to {index_path}")'''

In [ ]:
'''print("Logging data and index to Weights & Biases...")

artifact = wandb.Artifact(
    name="wikipedia_knowledge_base",
    type="dataset",
    description="JSONL chunks and FAISS index for Wikipedia Dataset",
    metadata={
        "chunking_strategy": "TokenChunker",
        "chunk_size_tokens": CONFIG["chunk_size"],
        "chunk_overlap_tokens": CONFIG["chunk_overlap"],
        "total_chunks_generated": len(chunk_records),
        
        "embedding_model": CONFIG["embedding_model"],
        "embedding_dimension": embedding_dimension,
        "vector_index_type": "FAISS IndexFlatL2",
        
        "source_material_type": "Wikipedia",
    }
)

artifact.add_file("/kaggle/working/knowledge_base.jsonl")
artifact.add_file("/kaggle/working/vector_index.idx")
wandb.log_artifact(artifact)

print("Chunking and Indexing complete and pushed to W&B")'''

In [ ]:
#wapi.artifact("23f2000391-dl-genai-project/23f2000391-t22026/wikipedia_knowledge_base:v0").download()

In [ ]:
'''jsonl_path="/kaggle/working/artifacts/wikipedia_knowledge_base:v0/knowledge_base.jsonl"
index_path="/kaggle/working/artifacts/wikipedia_knowledge_base:v0/vector_index.idx"'''

In [ ]:
'''print("Loading Knowledge Base")
chunks = []
with jsonlines.open(jsonl_path) as reader:
    for obj in reader:
        chunks.append(obj)

print("Loading FAISS Index...")
faiss_index = faiss.read_index(index_path)'''

In [ ]:
'''def rag_predict(row,kb,index):    
    row_embeddings=model.encode(row["clean_prompt"],convert_to_numpy=True,normalize_embeddings=True).reshape(1,-1)
    distances,indices=index.search(row_embeddings,k=20)

    results=[]
    for rank,(dist,idx) in enumerate(zip(distances[0],indices[0])):
        chunk_data=kb[idx]
        results.append({
            "rank": rank + 1,
            "distance": dist,
            "title": chunk_data["title"],
            "text": chunk_data["text"]
        })
            
    pairs=[[row["clean_prompt"],chunk['text']] for chunk in results] 
    ce_scores=cross_encoder.predict(pairs,batch_size=16) 

    for i,score in enumerate(ce_scores):
        results[i]["rerank_score"]=score

    reranked_chunks=sorted(results,key=lambda x: x["rerank_score"],reverse=True)
    top_3_chunks=reranked_chunks[:3]

    context_text="\n\n".join([f"Context {idx+1}\nTitle: {c['title']}\nContent: {c['text']}" for idx,c in enumerate(top_3_chunks)])

    query_row=f"""
    Retrieved Context
    {context_text}
    
    Question:
    {row['clean_prompt']}
    
    A. {row['A']}
    B. {row['B']}
    C. {row['C']}
    D. {row['D']}
    E. {row['E']}
    
    Return only the three most likely answer letters separated by spaces.
    """
    
    system_prompt="""
    You are an expert multiple-choice reasoning assistant.
    Use the retrieved context to help answer the question.
    If the context is insufficient, rely on your own reasoning.
    Return ONLY the top three answer letters separated by spaces.
    
    Example:
    A C D
    
    Do not explain your reasoning.
    """
    messages=[
        {"role": "system","content": system_prompt},
        {"role": "user","content": query_row}
    ]

    text_prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text_prompt,return_tensors="pt",truncation=True,max_length=4096).to(llm.device)
    
    with torch.no_grad():
        with torch.autocast("cuda"):
            outputs = llm.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
    
    response=tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    labels=re.findall(r"\b[A-E]\b", response.upper())
    labels=list(dict.fromkeys(labels))
    
    for letter in "ABCDE":
        if letter not in labels:
            labels.append(letter)
    
    return labels[:3]
'''

In [ ]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]
for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        top3_preds=rag_predict(row,chunks,faiss_index)        
        predictions.append(top3_preds)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })
'''

In [ ]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

In [ ]:
'''test_predictions=[]
for _, row in tqdm(test.iterrows(), total=len(test)):
    test_predictions.append(rag_predict(row,chunks,faiss_index))
test_predictions[:5]'''

### Train Dataset

In [106]:
model=SentenceTransformer('all-MiniLM-L6-v2') 
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [123]:
def build_kb(df):   
    kb=[]
    for idx, row in df.iterrows(): 
        kb.append({
            "prompt": row["clean_prompt"],
            "A": row["clean_A"],
            "B": row["clean_B"],
            "C": row["clean_C"],
            "D": row["clean_D"],
            "E": row["E"],
            "answer": row["answer"]
        })
    prompt_texts = [i["prompt"] for i in kb]

    kb_embeddings = model.encode(
        prompt_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    
    dim=kb_embeddings.shape[1]
    index=faiss.IndexFlatIP(dim)
    index.add(kb_embeddings)
    return kb,index

In [124]:
def retrieve(query,kb,index,k=20):
    emb=model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).reshape(1,-1)

    scores,indices=index.search(emb,k)

    retrieved=[]
    for score,idx in zip(scores[0],indices[0]):
        item=kb[idx].copy()
        item["faiss_score"]=float(score)
        retrieved.append(item)
    return retrieved

In [125]:
def rerank(query,retrieved):
    pairs=[[query,item["prompt"]] for item in retrieved]
    ce_scores=cross_encoder.predict(pairs)

    for i,s in enumerate(ce_scores):
        retrieved[i]["ce_score"]=float(s)
    retrieved=sorted(retrieved,key=lambda x:x["ce_score"],reverse=True)
    return retrieved

In [134]:
def rag_predict(row,kb,index):
    query=row["clean_prompt"]
    retrieved=retrieve(query,kb,index,k=50)
    retrieved=rerank(query,retrieved)
    top3=[]
    for i in retrieved:
        if i["answer"] not in top3:
            top3.append(i["answer"])
        if len(top3)==3:
            break
    
    return top3

In [159]:
fold_map3=[]
fold_acc=[]
fold_top3=[]
fold_f1=[]
for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    kb,index=build_kb(train_df)

    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        top3_preds=rag_predict(row,kb,index)        
        predictions.append(top3_preds)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)
    f1=macro_f1_score(val_df["answer"],predictions)
    
    print(f"MAP@3:{map3:.4f}")
    print(f"Accuracy:{acc:.4f}")
    print(f"Top3 Accuracy:{top3:.4f}")
    print(f"Macro F1:{f1:.4f}")
    
    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)
    fold_f1.append(f1)
    wandb.log({
        "Fold":fold+1,
        "MAP@3":map3,
        "Top1 Accuracy":acc,
        "Top3 Accuracy":top3,
        "Macro F1 Score":f1
    })

Fold: 1


100%|██████████| 400/400 [00:11<00:00, 36.36it/s]


MAP@3: 0.9975
Accuracy: 0.9950
Top3 Accuracy: 1.0000
Macro F1: 0.9949
Fold: 2


100%|██████████| 400/400 [00:11<00:00, 34.80it/s]


MAP@3: 0.9988
Accuracy: 0.9975
Top3 Accuracy: 1.0000
Macro F1: 0.9974
Fold: 3


...

MAP@3: 0.9975
Accuracy: 0.9950
Top3 Accuracy: 1.0000
Macro F1: 0.9947
Fold: 4


100%|██████████| 400/400 [00:11<00:00, 35.84it/s]


MAP@3: 0.9950
Accuracy: 0.9900
Top3 Accuracy: 1.0000
Macro F1: 0.9893
Fold: 5


100%|██████████| 400/400 [00:11<00:00, 35.76it/s]

MAP@3: 0.9988
Accuracy: 0.9975
Top3 Accuracy: 1.0000
Macro F1: 0.9974


In [163]:
print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))
print("Average Macro F1 Score:", np.mean(fold_f1))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3),
    "Average Macro F1 Score": np.mean(fold_f1)
})

wandb.finish()

Average MAP@3: 0.9975000000000002
Average Accuracy: 0.9949999999999999
Average Top3 Accuracy: 1.0
Average Macro F1 Score: 0.9947277476356049


'wandb.log({\n    "Average MAP@3": np.mean(fold_map3),\n    "Average Top1 Accuracy": np.mean(fold_acc),\n    "Average Top3 Accuracy": np.mean(fold_top3),\n    "Average Macro F1 Score": np.mean(fold_f1)\n})\n\nwandb.finish()'

In [136]:
test_predictions=[]
kb_train,index_train=build_kb(train)
for _, row in tqdm(test.iterrows(), total=len(test)):
    test_predictions.append(rag_predict(row,kb_train,index_train))
test_predictions[:5]

100%|██████████| 500/500 [00:13<00:00, 35.86it/s]


[['A', 'D', 'E'], ['B', 'C', 'D'], ['B'], ['E', 'D', 'C'], ['C', 'A', 'E']]

# Submission

In [137]:
submission_preds = [" ".join(pred) for pred in test_predictions]
sub=pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sub.set_index('ID',inplace=True)
sub.head()

,Prediction
ID,
1,A B C
2,A B C
3,A B C
4,A B C
5,A B C


In [138]:
sub['Prediction']=submission_preds
sub.head()

,Prediction
ID,
1,A D E
2,B C D
3,B
4,E D C
5,C A E


In [131]:
sub.to_csv("submission.csv")
print("Submission File Created")

Submission File Created
